In [0]:
!pip install faker
!pip install numpy pandas

In [0]:
import os
import random
import pandas as pd
from zoneinfo import ZoneInfo

from faker import Faker
from datetime import datetime, timedelta


fake = Faker("en_IN")

random.seed(42)
Faker.seed(42)

BASE_PATH = "/Volumes/private_volume/volume/raw_files"



CUSTOMER_PATH = f"{BASE_PATH}/customers"

ACCOUNT_PATH = f"{BASE_PATH}/accounts"

TRANSACTION_PATH = f"{BASE_PATH}/transactions"



INCOMING_CUSTOMER_PATH = f"{BASE_PATH}/incoming/customers"

INCOMING_ACCOUNT_PATH = f"{BASE_PATH}/incoming/accounts"

INCOMING_TRANSACTION_PATH = f"{BASE_PATH}/incoming/transactions"


CUSTOMER_INITIAL_FILE = f"{CUSTOMER_PATH}/customers.csv"

ACCOUNT_INITIAL_FILE = f"{ACCOUNT_PATH}/accounts.csv"

TRANSACTION_INITIAL_FILE = f"{TRANSACTION_PATH}/transactions.csv"


folders = [
    CUSTOMER_PATH,
    ACCOUNT_PATH,
    TRANSACTION_PATH,

    INCOMING_CUSTOMER_PATH,
    INCOMING_ACCOUNT_PATH,
    INCOMING_TRANSACTION_PATH
]

for path in folders:
    os.makedirs(path, exist_ok=True)

print("All generator folders created successfully!")


print("\nINITIAL DATA FILES")

print(CUSTOMER_INITIAL_FILE)
print(ACCOUNT_INITIAL_FILE)
print(TRANSACTION_INITIAL_FILE)

print("\nINCOMING DATA FOLDERS")

print(INCOMING_CUSTOMER_PATH)
print(INCOMING_ACCOUNT_PATH)
print(INCOMING_TRANSACTION_PATH)

In [0]:

def generate_customers(n):

    customers = []

    cities = [
        "Delhi", "Mumbai", "Bengaluru",
        "Hyderabad", "Chennai", "Pune",
        "Kolkata", "Jaipur"
    ]

    segments = ["Retail", "Premium", "Business"]
    kyc_statuses = ["Verified", "Pending", "Rejected"]

    for i in range(1, n + 1):

        customers.append({
            "customer_id": f"C{i:05d}",
            "name": fake.name(),
            "gender": random.choice(["Male", "Female"]),
            "email": fake.email(),
            "phone": fake.phone_number(),
            "dob": fake.date_of_birth(minimum_age=18, maximum_age=75),
            "city": random.choice(cities),
            "segment": random.choice(segments),
            "kyc_status": random.choice(kyc_statuses),
            "credit_score": random.randint(300, 850),
            "annual_income": random.randint(200000, 2500000),
            "join_date": fake.date_between(
                start_date="-5y",
                end_date="today"
            )
        })

    return pd.DataFrame(customers)


In [0]:

def generate_accounts(customers_df):

    accounts = []

    account_types = [
        "Savings", "Current", "Salary"
    ]

    branches = [
        "Delhi", "Mumbai", "Bengaluru",
        "Hyderabad", "Chennai", "Pune"
    ]

    statuses = [
        "Active", "Inactive", "Closed"
    ]

    account_counter = 1

    for customer_id in customers_df["customer_id"]:

        number_of_accounts = random.randint(1, 3)

        for _ in range(number_of_accounts):

            accounts.append({
                "account_id": f"A{account_counter:06d}",
                "customer_id": customer_id,
                "account_type": random.choice(account_types),
                "branch": random.choice(branches),
                "open_date": fake.date_between(
                    start_date="-5y",
                    end_date="today"
                ),
                "status": random.choice(statuses),
                "balance": round(
                    random.uniform(1000, 500000),
                    2
                ),
                "currency": "INR"
            })

            account_counter += 1

    return pd.DataFrame(accounts)


# -----------------------------
# Transactions
# -----------------------------

def generate_transactions(accounts_df, n):

    transactions = []

    txn_types = [
        "Credit", "Debit", "Transfer"
    ]

    channels = [
        "ATM", "Mobile", "Internet Banking",
        "Branch", "POS"
    ]

    merchant_categories = [
        "Shopping", "Food", "Travel",
        "Healthcare", "Utilities",
        "Entertainment", "Other"
    ]

    statuses = [
        "Success", "Failed", "Pending"
    ]

    account_ids = accounts_df["account_id"].tolist()

    for i in range(1, n + 1):

        transactions.append({
            "txn_id": f"T{i:08d}",
            "account_id": random.choice(account_ids),
            "txn_type": random.choice(txn_types),
            "txn_timestamp": fake.date_time_between(
                start_date="-1y",
                end_date="now"
            ),
            "channel": random.choice(channels),
            "merchant_category": random.choice(
                merchant_categories
            ),
            "amount": round(
                random.uniform(100, 100000),
                2
            ),
            "status": random.choice(statuses),
            "fraud_flag": random.choice([0, 0, 0, 0, 1])
        })

    return pd.DataFrame(transactions)


# Generate initial data
customers_df = generate_customers(500)

accounts_df = generate_accounts(customers_df)

transactions_df = generate_transactions(
    accounts_df,
    n=5000
)

# Add CDC metadata to initial records so initial and incoming files share columns.
initial_event_time = datetime.now(ZoneInfo("Asia/Kolkata"))
for _df in (customers_df, accounts_df, transactions_df):
    _df["operation"] = "INSERT"
    _df["event_timestamp"] = initial_event_time

print("Customers:", len(customers_df))
print("Accounts:", len(accounts_df))
print("Transactions:", len(transactions_df))



In [0]:

PLACEHOLDERS = [
    "N/A",
    "NULL",
    "null",
    "NA",
    "-",
    "unknown",
    "?",
    "None",
    "#N/A"
]

DATE_PLACEHOLDERS = [
    "1900-01-01",
    "0000-00-00"
]

DATE_FMTS = [
    "%Y-%m-%d",
    "%d/%m/%Y",
    "%d-%b-%Y",
    "%m/%d/%Y",
    "%Y%m%d"
]

DATETIME_FMTS = [
    "%Y-%m-%d %H:%M:%S",
    "%d/%m/%Y %H:%M",
    "%m-%d-%Y %H:%M:%S",
    "%Y/%m/%d %H:%M:%S"
]


# ---------------------------------------------------------
# MIXED DATE FORMATS
# ---------------------------------------------------------

def _mixed_dates(rng, s, fmts):

    dt = pd.to_datetime(
        s,
        errors="coerce"
    )

    pick = rng.integers(
        0,
        len(fmts),
        len(s)
    )

    out = np.empty(
        len(s),
        dtype=object
    )

    for i, f in enumerate(fmts):

        m = pick == i

        out[m] = dt[m].dt.strftime(
            f
        ).to_numpy(dtype=object)

    return pd.Series(
        out,
        dtype=object
    )


# ---------------------------------------------------------
# MESSY TEXT
# ---------------------------------------------------------

def _messy_text(rng, s):

    arr = s.to_numpy(
        dtype=object
    ).copy()

    r = rng.random(
        len(arr)
    )

    pad = r < 0.03

    up = (
        (r >= 0.03)
        & (r < 0.06)
    )

    low = (
        (r >= 0.06)
        & (r < 0.09)
    )

    arr[pad] = [
        f" {x} "
        for x in arr[pad]
    ]

    arr[up] = [
        str(x).upper()
        for x in arr[up]
    ]

    arr[low] = [
        str(x).lower()
        for x in arr[low]
    ]

    return pd.Series(
        arr,
        dtype=object
    )


# ---------------------------------------------------------
# MESSY NUMBERS
# ---------------------------------------------------------

def _messy_numbers(rng, s, is_float):

    arr = s.to_numpy(
        dtype=object
    ).copy()

    if is_float:

        vals = s.astype(
            float
        ).to_numpy()

        fmt = (
            rng.random(len(arr))
            < 0.01
        )

        arr[fmt] = [
            f"{v:,.2f}"
            for v in vals[fmt]
        ]

        out = (
            (rng.random(len(arr)) < 0.003)
            & ~fmt
        )

        arr[out] = vals[out] * 100

    return pd.Series(
        arr,
        dtype=object
    )


# ---------------------------------------------------------
# APPLY MESSINESS
# ---------------------------------------------------------

def make_messy(
    df,
    rng,
    key_cols=(),
    date_cols=(),
    datetime_cols=(),
    dup_rate=0.01
):

    """Convert clean data into a messy raw extract."""

    df = df.copy().reset_index(
        drop=True
    )

    n = len(df)

    for col in df.columns:

        if col in key_cols:
            continue

        dtype = df[col].dtype

        placeholders = list(
            PLACEHOLDERS
        )

        # Dates
        if (
            col in date_cols
            or col in datetime_cols
        ):

            fmts = (
                DATETIME_FMTS
                if col in datetime_cols
                else DATE_FMTS
            )

            s = _mixed_dates(
                rng,
                df[col],
                fmts
            )

            placeholders += DATE_PLACEHOLDERS

        # Boolean
        elif pd.api.types.is_bool_dtype(dtype):

            reps = np.array(
                [
                    ["True", "False"],
                    ["Y", "N"],
                    ["1", "0"],
                    ["true", "false"],
                    ["yes", "no"]
                ],
                dtype=object
            )

            pick = rng.integers(
                0,
                len(reps),
                n
            )

            s = pd.Series(
                np.where(
                    df[col].to_numpy(),
                    reps[pick, 0],
                    reps[pick, 1]
                ),
                dtype=object
            )

        # Numeric
        elif pd.api.types.is_numeric_dtype(dtype):

            s = _messy_numbers(
                rng,
                df[col],
                pd.api.types.is_float_dtype(dtype)
            )

        # Text
        else:

            s = _messy_text(
                rng,
                df[col].astype(object)
            )

        # Random dirtiness rates
        null_r = rng.uniform(
            0.01,
            0.05
        )

        blank_r = rng.uniform(
            0.005,
            0.02
        )

        ph_r = rng.uniform(
            0.003,
            0.015
        )

        r = rng.random(n)

        arr = s.to_numpy(
            dtype=object
        ).copy()

        m_null = r < null_r

        m_blank = (
            (r >= null_r)
            & (r < null_r + blank_r)
        )

        m_ph = (
            (r >= null_r + blank_r)
            & (
                r
                < null_r + blank_r + ph_r
            )
        )

        # Nulls
        arr[m_null] = None

        # Whitespace blanks
        arr[m_blank] = rng.choice(
            [" ", "  ", "   "],
            m_blank.sum()
        )

        # Placeholder values
        arr[m_ph] = rng.choice(
            placeholders,
            m_ph.sum()
        )

        df[col] = arr

    # Duplicate records
    dups = df.sample(
        frac=dup_rate,
        random_state=int(
            rng.integers(
                0,
                2**31 - 1
            )
        )
    )

    df = pd.concat(
        [df, dups]
    )

    # Shuffle records
    df = df.sample(
        frac=1,
        random_state=int(
            rng.integers(
                0,
                2**31 - 1
            )
        )
    ).reset_index(
        drop=True
    )

    return df


print("Messy-data functions defined")


In [0]:
customers_df.to_csv(
    f"{CUSTOMER_PATH}/customers.csv",
    index=False
)

accounts_df.to_csv(
    f"{ACCOUNT_PATH}/accounts.csv",
    index=False
)

transactions_df.to_csv(
    f"{TRANSACTION_PATH}/transactions.csv",
    index=False
)

print("Initial files saved")
def generate_customer_events(
    customers_df,
    number_of_inserts=random.randint(3, 15),
    number_of_updates=random.randint(25, 100)
):

    events = []

    now = datetime.now()

    # -----------------------------
    # New customer INSERT events
    # -----------------------------

    max_customer_number = len(customers_df)

    for i in range(1, number_of_inserts + 1):

        customer_number = max_customer_number + i

        events.append({
            "customer_id": f"C{customer_number:05d}",
            "name": fake.name(),
            "gender": random.choice(["Male", "Female"]),
            "email": fake.email(),
            "phone": fake.phone_number(),
            "dob": fake.date_of_birth(
                minimum_age=18,
                maximum_age=75
            ),
            "city": random.choice([
                "Delhi", "Mumbai", "Bengaluru",
                "Hyderabad", "Chennai", "Pune"
            ]),
            "segment": random.choice([
                "Retail", "Premium", "Business"
            ]),
            "kyc_status": random.choice([
                "Verified", "Pending", "Rejected"
            ]),
            "credit_score": random.randint(300, 850),
            "annual_income": random.randint(
                200000, 2500000
            ),
            "join_date": now.date(),
            "operation": "INSERT",
            "event_timestamp": now
        })

    # -----------------------------
    # Existing customer UPDATE events
    # -----------------------------

    existing_customers = customers_df.sample(
        n=min(number_of_updates, len(customers_df))
    )

    for _, customer in existing_customers.iterrows():

        events.append({
            "customer_id": customer["customer_id"],
            "name": customer["name"],
            "gender": customer["gender"],
            "email": customer["email"],
            "phone": customer["phone"],
            "dob": customer["dob"],

            # Change customer attributes
            "city": random.choice([
                "Delhi", "Mumbai", "Bengaluru",
                "Hyderabad", "Chennai", "Pune"
            ]),

            "segment": random.choice([
                "Retail", "Premium", "Business"
            ]),

            "kyc_status": random.choice([
                "Verified", "Pending", "Rejected"
            ]),

            "credit_score": random.randint(300, 850),
            "annual_income": random.randint(
                200000, 2500000
            ),

            "join_date": customer["join_date"],

            "operation": "UPDATE",
            "event_timestamp": now
        })

    return pd.DataFrame(events)


customer_events_df = generate_customer_events(
    customers_df,
    number_of_inserts=5,
    number_of_updates=10
)

customer_events_df.head()
batch_id = datetime.now(
    ZoneInfo("Asia/Kolkata")
).strftime("%Y%m%d_%H%M%S")

print(batch_id)

customer_file = (
    f"{INCOMING_CUSTOMER_PATH}/"
    f"customers_{batch_id}.csv"
)

customer_events_df.to_csv(
    customer_file,
    index=False
)

print("Customer CDC file created:")
print(customer_file)
def generate_account_events(
    customers_df,
    accounts_df,
    number_of_inserts=random.randint(3, 15),
    number_of_updates=random.randint(3, 15)
):

    events = []

    now = datetime.now()

    max_account_number = len(accounts_df)

    # -----------------------------
    # New account INSERT events
    # -----------------------------

    for i in range(1, number_of_inserts + 1):

        account_number = max_account_number + i

        customer_id = random.choice(
            customers_df["customer_id"].tolist()
        )

        events.append({
            "account_id": f"A{account_number:06d}",
            "customer_id": customer_id,
            "account_type": random.choice([
                "Savings", "Current", "Salary"
            ]),
            "branch": random.choice([
                "Delhi", "Mumbai", "Bengaluru",
                "Hyderabad", "Chennai", "Pune"
            ]),
            "open_date": now.date(),
            "status": "Active",
            "balance": round(
                random.uniform(1000, 500000),
                2
            ),
            "currency": "INR",
            "operation": "INSERT",
            "event_timestamp": now
        })

    # -----------------------------
    # Existing account UPDATE events
    # -----------------------------

    existing_accounts = accounts_df.sample(
        n=min(number_of_updates, len(accounts_df))
    )

    for _, account in existing_accounts.iterrows():

        events.append({
            "account_id": account["account_id"],
            "customer_id": account["customer_id"],
            "account_type": account["account_type"],
            "branch": account["branch"],
            "open_date": account["open_date"],

            # Simulate account status change
            "status": random.choice([
                "Active", "Inactive", "Closed"
            ]),

            # Simulate balance change
            "balance": round(
                random.uniform(1000, 500000),
                2
            ),

            "currency": account["currency"],

            "operation": "UPDATE",
            "event_timestamp": now
        })

    return pd.DataFrame(events)


account_events_df = generate_account_events(
    customers_df,
    accounts_df,
    number_of_inserts=random.randint(3, 15),
    number_of_updates=random.randint(3, 50)
)

account_events_df.head()
account_file = (
    f"{INCOMING_ACCOUNT_PATH}/"
    f"accounts_{batch_id}.csv"
)

account_events_df.to_csv(
    account_file,
    index=False
)

print("Account CDC file created:")
print(account_file)
def generate_transaction_events(
    accounts_df,
    n
):

    transactions = []

    now = datetime.now()

    account_ids = accounts_df["account_id"].tolist()

    for i in range(1, n + 1):

        transactions.append({
            "txn_id": (
                f"TXN_{batch_id}_{i:05d}"
            ),

            "account_id": random.choice(account_ids),

            "txn_type": random.choice([
                "Credit", "Debit", "Transfer"
            ]),

            "txn_timestamp": now,

            "channel": random.choice([
                "ATM", "Mobile", "Internet Banking",
                "Branch", "POS"
            ]),

            "merchant_category": random.choice([
                "Shopping", "Food", "Travel",
                "Healthcare", "Utilities",
                "Entertainment", "Other"
            ]),

            "amount": round(
                random.uniform(100, 100000),
                2
            ),

            "status": random.choice([
                "Success", "Failed", "Pending"
            ]),

            "fraud_flag": random.choice([
                0, 0, 0, 0, 1
            ]),

            "operation": "INSERT",
            "event_timestamp": now
        })

    return pd.DataFrame(transactions)


transaction_events_df = generate_transaction_events(
    accounts_df,
    n=1000
)

transaction_events_df.head()
transaction_file = (
    f"{INCOMING_TRANSACTION_PATH}/"
    f"transactions_{batch_id}.csv"
)

transaction_events_df.to_csv(
    transaction_file,
    index=False
)

print("Transaction event file created:")
print(transaction_file)
print("Initial source files:")
print(os.listdir(CUSTOMER_PATH))
print(os.listdir(ACCOUNT_PATH))
print(os.listdir(TRANSACTION_PATH))

print("\nIncoming customer files:")
print(os.listdir(INCOMING_CUSTOMER_PATH))

print("\nIncoming account files:")
print(os.listdir(INCOMING_ACCOUNT_PATH))

print("\nIncoming transaction files:")
print(os.listdir(INCOMING_TRANSACTION_PATH))

transaction_events_df.to_csv(
    transaction_file,
    index=False
)

print("Transaction CDC file created:")
print(transaction_file)

print("\\nGenerator completed successfully.")

